###Формирование сетки и L-образной области

Здесь создаётся вычислительная сетка и задаётся форма мембраны.

1. **Сетка и координаты**

   Формируем равномерную сетку по x и y, а также матрицы координат:
   $$
   X_{ij} = x_i, \qquad Y_{ij} = y_j.
   $$

2. **Маска области**

   Булева маска `inside` описывает геометрию мембраны:
   - `True` — точка принадлежит области,
   - `False` — точка вырезана.

   L-образная область получается вырезанием прямоугольника:
   $$
   0.5 \le x \le 1,\qquad 0 \le y \le 0.5.
   $$

3. **Визуализация**

   Показ маски позволяет убедиться в корректности геометрии.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

nx = ny = 61

# Координатная сетка на квадрате [0,1] x [0,1]
x = np.linspace(0.0, 1.0, nx)
y = np.linspace(0.0, 1.0, ny)
X, Y = np.meshgrid(x, y, indexing="ij")

h = x[1] - x[0]  # шаг

# Массив inside: True - точка принадлежит мембране, False - "дырка"
inside = np.ones_like(X, dtype=bool)

# Задаём L-образную область:
inside[(X >= 0.5) & (Y <= 0.5)] = False

plt.figure(figsize=(4,4))
plt.title("L-образная мембрана (маска inside)")
plt.imshow(inside.T, origin="lower", extent=[0,1,0,1])
plt.xlabel("x")
plt.ylabel("y")
plt.colorbar(label="1 - внутри, 0 - вне")
plt.show()


###Определение границ и внутренних точек

Здесь определяется, какие узлы являются:
— внутренними,  
— граничными (где фиксируется \( u = 0 \)),  
— вне мембраны.

1. **Классификация точек**

   Граница определяется по наличию соседей вне области.

   Внутренние точки:
   $$
   \text{is\_interior} = \text{inside} \cap \neg \text{is\_boundary}.
   $$

2. **Внутренние углы**

   Реэнтрантный угол L-области автоматически распознаётся как граница, т.к. у него есть соседние точки вне мембраны.

3. **Визуализация**

   Показывается карта:
   - 0 — вне области,
   - 1 — граница,
   - 2 — внутренние точки.


In [ ]:

is_boundary = np.zeros_like(inside, dtype=bool)

for i in range(nx):
    for j in range(ny):
        if not inside[i, j]:
            # Вне мембраны — нас не интересует
            continue

        if i == 0 or i == nx - 1 or j == 0 or j == ny - 1:
            is_boundary[i, j] = True
            continue

        # Внутренние границы (ребро вырезанного прямоугольника)
        if (i > 0   and not inside[i-1, j]) or \
           (i < nx-1 and not inside[i+1, j]) or \
           (j > 0   and not inside[i, j-1]) or \
           (j < ny-1 and not inside[i, j+1]):
            is_boundary[i, j] = True

is_interior = inside & (~is_boundary)

print("Всего узлов в мембране:", inside.sum())
print("Граничных узлов:", is_boundary.sum())
print("Внутренних узлов:", is_interior.sum())

domain_vis = np.zeros_like(inside, dtype=int)
domain_vis[inside] = 2
domain_vis[is_boundary] = 1
domain_vis[~inside] = 0

plt.figure(figsize=(4,4))
plt.title("Кодировка: 0 - вне, 1 - граница, 2 - внутренние")
plt.imshow(domain_vis.T, origin="lower", extent=[0,1,0,1])
plt.xlabel("x")
plt.ylabel("y")
plt.colorbar()
plt.show()


###Задание распределённой нагрузки \(P(x,y)\)

Вводится распределённая нагрузка на мембрану.

1. **Гауссова нагрузка**

   Локализованная нагрузка задаётся формулой:
   $$
   P(x,y) = P_0 \exp\left(
   -\frac{(x - x_0)^2 + (y - y_0)^2}{2\sigma^2}
   \right).
   $$

   Такой источник нагрузки имитирует локальное давление.

2. **Зануление вне мембраны**

   На вырезанной части:
   $$
   P = 0.
   $$

3. **Графическое отображение**

   Показ распределения нагрузки, включая центр пика.


In [ ]:
# Постоянное натяжение мембраны
T = 1.0

# Параметры гауссовой нагрузки
P0 = 1.0
x0, y0 = 0.25, 0.75
sigma = 0.08

# Гауссов пик
P = P0 * np.exp(-((X - x0)**2 + (Y - y0)**2) / (2 * sigma**2))

# Вне мембраны нагрузка не действует
P[~inside] = 0.0

plt.figure(figsize=(4,4))
plt.title("Нагрузка P(x,y)")
plt.imshow(P.T, origin="lower", extent=[0,1,0,1])
plt.xlabel("x")
plt.ylabel("y")
plt.colorbar(label="P")
plt.scatter([x0], [y0], color="red", label="центр гаусса")
plt.legend()
plt.show()


###Решение уравнения Пуассона методом Гаусса–Зейделя

Решается уравнение стационарного прогиба:

$$
\nabla^2 u = -\frac{P}{T}.
$$

1. **Дискретизация лапласиана**

   Для внутреннего узла:
   $$
   \frac{u_{i+1,j} + u_{i-1,j} + u_{i,j+1} + u_{i,j-1} - 4u_{i,j}}{h^2}
   = -\frac{P_{ij}}{T}.
   $$

2. **Итерационная формула Гаусса–Зейделя**
   $$
   u_{i,j}^{(new)} =
   \frac{
      u_{i+1,j} + u_{i-1,j}
      + u_{i,j+1} + u_{i,j-1}
      + h^2 \frac{P_{ij}}{T}
   }{4}.
   $$

3. **Граничные условия**

   На всех границах:
   $$
   u = 0.
   $$

4. **Критерий остановки**

   Итерации продолжаются, пока:
   $$
   \max|u^{new} - u^{old}| < 10^{-5}.
   $$


In [ ]:
# Поле прогиба
u = np.zeros_like(X, dtype=float)

max_iter = 5000
tol = 1e-5

for it in range(max_iter):
    max_diff = 0.0

    for i in range(1, nx - 1):
        for j in range(1, ny - 1):
            if not is_interior[i, j]:
                continue

            u_old = u[i, j]

            u[i, j] = 0.25 * (
                u[i+1, j] + u[i-1, j] +
                u[i, j+1] + u[i, j-1] +
                (h**2) * P[i, j] / T
            )

            diff = abs(u[i, j] - u_old)
            if diff > max_diff:
                max_diff = diff

    if it % 50 == 0:
        print(f"Итерация {it}, max_diff = {max_diff:.2e}")

    if max_diff < tol:
        print(f"Сошлось на итерации {it}, max_diff = {max_diff:.2e}")
        break
else:
    print("Достигнут предел итераций без достижения заданной точности.")


###Визуализация прогиба и построение сечений

Показ геометрии прогиба и его профилей.

1. **3D-поверхность прогиба**

   Отображается поверхность:
   $$
   u(x,y),
   $$
   где вне области:
   $$
   u = \mathrm{NaN}.
   $$

2. **Сечение вдоль оси x**

   Сечение проходит через точку нагрузки:
   $$
   u(x, y_0).
   $$

3. **Сечение вдоль оси y**
   $$
   u(x_0, y).
   $$

Эти графики позволяют оценить влияние формы мембраны на прогиб.


In [ ]:

from mpl_toolkits.mplot3d import Axes3D

u_plot = np.where(inside, u, np.nan)

fig = plt.figure(figsize=(7,5))
ax = fig.add_subplot(111, projection='3d')
ax.set_title("Прогиб мембраны u(x,y)")
ax.plot_surface(X, Y, u_plot, rstride=1, cstride=1, linewidth=0)
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("u")
plt.show()

# --- Сечения вдоль осей x и y через центр нагрузки (x0, y0) ---
ix = np.argmin(np.abs(x - x0))
iy = np.argmin(np.abs(y - y0))

u_x = u[:, iy].copy()
for i in range(nx):
    if not inside[i, iy]:
        u_x[i] = np.nan

u_y = u[ix, :].copy()
for j in range(ny):
    if not inside[ix, j]:
        u_y[j] = np.nan

fig, axes = plt.subplots(1, 2, figsize=(10,4))

axes[0].plot(x, u_x, marker="o")
axes[0].set_title(f"Сечение u(x, y={y[iy]:.2f})")
axes[0].set_xlabel("x")
axes[0].set_ylabel("u")

axes[1].plot(y, u_y, marker="o")
axes[1].set_title(f"Сечение u(x={x[ix]:.2f}, y)")
axes[1].set_xlabel("y")
axes[1].set_ylabel("u")

plt.tight_layout()
plt.show()
